In [5]:
# ============================================
# نسخه نهایی - برگشت به روش اولیه با بهبودها
# ============================================

# =========================
# 1️⃣ Mount Drive
# =========================
from google.colab import drive
drive.mount('/content/drive')

# =========================
# 2️⃣ Extract Dataset
# =========================
import zipfile
import os

ZIP_PATH = "/content/drive/MyDrive/dataset_cancer_v1.zip"
EXTRACT_PATH = "/content"

if not os.path.exists("/content/dataset_cancer_v1"):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("Extraction completed.")
else:
    print("Dataset already extracted.")

# =========================
# 3️⃣ Set BASE_DIR
# =========================
BASE_DIR = "/content/dataset_cancer_v1/classificacao_binaria"
print("BASE_DIR =", BASE_DIR)

# =========================
# 4️⃣ Imports
# =========================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =========================
# 5️⃣ Configurations
# =========================
IMG_SIZE = 224
EPOCHS = 30
BATCH_SIZE = 32

label_map = {"benign": 0, "malignant": 1}
magnifications = ["40X", "100X", "200X", "400X"]

# =========================
# 6️⃣ Collect Image Paths
# =========================
filepaths = []
labels = []

for mag in magnifications:
    for cls in label_map.keys():
        folder = os.path.join(BASE_DIR, mag, cls)
        if not os.path.exists(folder):
            continue
        for img_name in os.listdir(folder):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                filepaths.append(os.path.join(folder, img_name))
                labels.append(label_map[cls])

filepaths = np.array(filepaths)
labels = np.array(labels)

print("Total samples:", len(filepaths))
print("Class distribution:")
print(pd.Series(labels).value_counts())

# =========================
# 7️⃣ Train / Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    filepaths,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

train_df = pd.DataFrame({"filename": X_train, "class": y_train})
test_df = pd.DataFrame({"filename": X_test, "class": y_test})

print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

# =========================
# 8️⃣ Class-specific Augmentation (مثل کد اولیه)
# =========================
benign_df = train_df[train_df['class'] == 0]
malignant_df = train_df[train_df['class'] == 1]

benign_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode="nearest"
)

malignant_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.05,
    horizontal_flip=True,
    fill_mode="nearest"
)

benign_gen = benign_datagen.flow_from_dataframe(
    benign_df,
    x_col="filename",
    y_col="class",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=True
)

malignant_gen = malignant_datagen.flow_from_dataframe(
    malignant_df,
    x_col="filename",
    y_col="class",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=True
)

# ترکیب دو ژنراتور
def combined_generator(gen1, gen2):
    while True:
        try:
            X1, y1 = next(gen1)
            X2, y2 = next(gen2)
            X = np.concatenate([X1, X2], axis=0)
            y = np.concatenate([y1, y2], axis=0)
            idx = np.arange(len(X))
            np.random.shuffle(idx)
            yield X[idx], y[idx]
        except StopIteration:
            # اگر یکی از ژنراتورها تموم شد، دوباره شروع کن
            continue

# تست ژنراتور
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_dataframe(
    test_df,
    x_col="filename",
    y_col="class",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False
)

steps_per_epoch = (len(benign_df) + len(malignant_df)) // BATCH_SIZE

# =========================
# 9️⃣ Build CNN (مدل ساده‌تر ولی با عمق بیشتر)
# =========================
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(256, (3,3), activation='relu'),  # لایه جدید
    MaxPooling2D(2,2),  # لایه جدید

    Flatten(),
    Dense(256, activation='relu'),  # افزایش
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# =========================
# 🔟 Callbacks
# =========================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

callbacks = [early_stop, reduce_lr]

# =========================
# 1️⃣1️⃣ Train
# =========================
print("\n" + "="*50)
print("🚀 Training CNN from scratch")
print("="*50)

history = model.fit(
    combined_generator(benign_gen, malignant_gen),
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=test_generator,
    validation_steps=len(test_df) // BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

# =========================
# 1️⃣2️⃣ Evaluation
# =========================
print("\n" + "="*50)
print("📊 FINAL EVALUATION")
print("="*50)

y_pred_prob = model.predict(test_generator)
y_pred = (y_pred_prob > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Benign", "Malignant"]))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

test_loss, test_acc = model.evaluate(test_generator, verbose=0)
print(f"\n✅ Final Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

model.save("/content/breast_cancer_cnn_binary.keras")
print("\n💾 Model saved as 'breast_cancer_cnn_binary.keras'")

print("\n🎉 Training Complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset already extracted.
BASE_DIR = /content/dataset_cancer_v1/classificacao_binaria
Total samples: 7909
Class distribution:
1    5429
0    2480
Name: count, dtype: int64
Train samples: 6327
Test samples: 1582
Found 1984 validated image filenames.
Found 4343 validated image filenames.
Found 1582 validated image filenames.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,858,881 (37.61 MB)

 Trainable params: 9,858,881 (37.61 MB)

 Non-trainable params: 0 (0.00 B)


🚀 Training CNN from scratch
Epoch 1/30
197/197 ━━━━━━━━━━━━━━━━━━━━ 205s 977ms/step - accuracy: 0.7716 - loss: 0.5106 - val_accuracy: 0.8374 - val_loss: 0.4499 - learning_rate: 1.0000e-04
Epoch 2/30
197/197 ━━━━━━━━━━━━━━━━━━━━ 184s 938ms/step - accuracy: 0.8089 - loss: 0.4567 - val_accuracy: 0.7838 - val_loss: 0.5319 - learning_rate: 1.0000e-04
Epoch 3/30
197/197 ━━━━━━━━━━━━━━━━━━━━ 186s 948ms/step - accuracy: 0.8217 - loss: 0.4234 - val_accuracy: 0.8559 - val_loss: 0.3762 - learning_rate: 1.0000e-04
Epoch 4/30
197/197 ━━━━━━━━━━━━━━━━━━━━ 184s 939ms/step - accuracy: 0.8326 - loss: 0.3915 - val_accuracy: 0.8622 - val_loss: 0.3674 - learning_rate: 1.0000e-04
Epoch 5/30
197/197 ━━━━━━━━━━━━━━━━━━━━ 184s 939ms/step - accuracy: 0.8375 - loss: 0.3776 - val_accuracy: 0.8654 - val_loss: 0.3306 - learning_rate: 1.0000e-04
Epoch 6/30
197/197 ━━━━━━━━━━━━━━━━━━━━ 183s 931ms/step - accuracy: 0.8481 - loss: 0.3527 - val_accuracy: 0.8584 - val_loss: 0.3432 - learning_rate: 1.0000e-04
Epoch 7/30
